## Importar datos desde Kaggle

In [0]:
!pip install kaggle

In [0]:
dbutils.library.restartPython()

In [0]:
%sh
# Sustituir las variables con el nombre de usuario y el token de Kaggle.
export KAGGLE_USERNAME="-----"
export KAGGLE_KEY="---"

In [0]:
%sh
cd /databricks/driver
pwd
kaggle datasets download -d ealtman2019/credit-card-transactions

In [0]:
%sh
cd /databricks/driver
unzip -o credit-card-transactions.zip -d credit-card-transactions/
ls -l /databricks/driver/credit-card-transactions/

In [0]:
dbutils.fs.cp(
    "file:/databricks/driver/credit-card-transactions/credit_card_transactions-ibm_v2.csv",
    "dbfs:/FileStore/credit_card_transactions-ibm_v2.csv")

## Create raw layer of transaction data

In [0]:
raw_transactions = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("dbfs:/FileStore/transacciones_completo/credit_card_transactions-ibm_v2.csv")
)

In [0]:
raw_transactions.limit(5).display()

User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,2002,9,1,2024-09-11T06:21:00Z,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,null,No
0,0,2002,9,1,2024-09-11T06:42:00Z,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,2,2024-09-11T06:22:00Z,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,2,2024-09-11T17:45:00Z,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,null,No
0,0,2002,9,3,2024-09-11T06:23:00Z,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,null,No


In [0]:
invalid_chars = [' ', ',', ';', '{', '}', '(', ')', '\n', '\t', '=']
invalid_columns = [col for col in raw_transactions.columns if any(char in col for char in invalid_chars)]
print(invalid_columns)

['Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Is Fraud?']


In [0]:
raw_transactions = (
    raw_transactions
    .withColumnRenamed('Use Chip', 'UseChip')
    .withColumnRenamed('Merchant Name', 'MerchantName')
    .withColumnRenamed('Merchant City', 'MerchantCity')
    .withColumnRenamed('Use Chip', 'UseChip')
    .withColumnRenamed('Merchant State', 'MerchantState')
    .withColumnRenamed('Errors?', 'Errors')
    .withColumnRenamed('Is Fraud?', 'IsFraud')
)

In [0]:
raw_transactions.limit(5).display()

User,Card,Year,Month,Day,Time,Amount,UseChip,MerchantName,MerchantCity,MerchantState,Zip,MCC,Errors,IsFraud?
0,0,2002,9,1,2024-09-11T06:21:00Z,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,null,No
0,0,2002,9,1,2024-09-11T06:42:00Z,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,2,2024-09-11T06:22:00Z,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,null,No
0,0,2002,9,2,2024-09-11T17:45:00Z,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,null,No
0,0,2002,9,3,2024-09-11T06:23:00Z,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,null,No


In [0]:
raw_transactions.write.format('delta').mode('overwrite').saveAsTable('raw_transactions')

# 